In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/driver_race_base.csv")
print(df.shape)
df.head()

(25121, 14)


,season,round,raceId,race_date,circuitId,circuit_name,driverId,driver_name,constructorId,constructor_name,grid,positionOrder,statusId,target_top10
0,1950,1,833,1950-05-13,9,Silverstone Circuit,579,Juan Fangio,51,Alfa Romeo,3,12,44,0
1,1950,1,833,1950-05-13,9,Silverstone Circuit,589,Louis Chiron,105,Maserati,11,18,8,0
2,1950,1,833,1950-05-13,9,Silverstone Circuit,619,Bob Gerard,151,ERA,13,6,13,1
3,1950,1,833,1950-05-13,9,Silverstone Circuit,627,Louis Rosier,154,Talbot-Lago,9,5,12,1
4,1950,1,833,1950-05-13,9,Silverstone Circuit,640,Toulo de Graffenried,105,Maserati,8,17,5,0


### ✅ Datum parsen & chronologisch sortieren


In [4]:
df["race_date"] = pd.to_datetime(df["race_date"], errors="coerce")

df = df.sort_values(
    ["season", "round", "race_date", "raceId", "driverId"]
).reset_index(drop=True)

df[["season", "round", "race_date", "raceId"]].head()

,season,round,race_date,raceId
0,1950,1,1950-05-13,833
1,1950,1,1950-05-13,833
2,1950,1,1950-05-13,833
3,1950,1,1950-05-13,833
4,1950,1,1950-05-13,833


In [5]:
# Ensure deterministic chronological order before any "past N races" features
df = df.sort_values(["season", "round", "race_date"]).reset_index(drop=True)


add feauture importance and plot

### Feature 1: Fahrer-Erfahrung


In [6]:
df = df.sort_values(["driverId", "season", "round", "race_date"]).reset_index(drop=True)
df["driver_experience"] = df.groupby("driverId").cumcount()


each year the experience grows 

### Feature 2: Aktuelle Fahrerform

In [7]:
N_FORM = 5

df = df.sort_values(["driverId", "season", "round", "race_date"]).reset_index(drop=True)

df["driver_recent_form"] = (
    df.groupby("driverId")["positionOrder"]
      .apply(lambda s: s.shift(1).rolling(N_FORM, min_periods=1).mean())
      .reset_index(level=0, drop=True)
)


### Feature 3: Teamstärke

In [8]:
N_TEAM = 10

df = df.sort_values(["constructorId", "season", "round", "race_date"]).reset_index(drop=True)

df["constructor_strength"] = (
    df.groupby("constructorId")["positionOrder"]
      .apply(lambda s: s.shift(1).rolling(N_TEAM, min_periods=1).mean())
      .reset_index(level=0, drop=True)
)


### Feature 4: Era-Bucket

In [11]:
def era_bucket(year):
    if year < 1970: return "1950-1969"
    if year < 1990: return "1970-1989"
    if year < 2010: return "1990-2009"
    if year < 2022: return "2010-2021"
    return "2022+"

df["era"] = df["season"].apply(era_bucket)
df["era"].value_counts()


era
1990-2009    7528
1970-1989    7388
2010-2021    5032
1950-1969    3854
2022+        1319
Name: count, dtype: int64

keep the exact years instead of bins

### Feature 5: Startpositions-Bucket

lost data because of bins  
use coordinates based on f1 logic 

In [12]:
df["grid_bucket"] = pd.cut(
    df["grid"],
    bins=[0, 3, 10, 20, 60],
    labels=["front_row_1_3", "top10_4_10", "mid_11_20", "back_21+"]
)

df["grid_bucket"].value_counts(dropna=False)

grid_bucket
mid_11_20        10862
top10_4_10        7915
front_row_1_3     3391
back_21+          2953
Name: count, dtype: int64

add perfomance indicator for driver and team (avg in time bucket)

### 🔗 Zusammenstellung des Feature-Datensatzes

In [13]:
feature_df = df[[
    "season", "round", "raceId", "race_date",          # keep only if you need them for splitting/holdout logic
    "grid",
    "driver_experience",
    "driver_recent_form",
    "constructor_strength",
    "grid_bucket",
    "target_top10"
]].copy()


print(feature_df.isna().mean().sort_values(ascending=False).head(10))
feature_df.head()

driver_recent_form      0.031527
constructor_strength    0.007882
season                  0.000000
round                   0.000000
raceId                  0.000000
race_date               0.000000
grid                    0.000000
driver_experience       0.000000
grid_bucket             0.000000
target_top10            0.000000
dtype: float64


,season,round,raceId,race_date,grid,driver_experience,driver_recent_form,constructor_strength,grid_bucket,target_top10
0,1968,8,674,1968-08-04,16,83,9.6,NaN,mid_11_20,0
1,1968,11,677,1968-10-06,5,36,3.6,13.0,top10_4_10,1
2,1971,1,632,1971-03-06,7,60,9.0,10.5,top10_4_10,1
3,1971,1,632,1971-03-06,11,7,10.4,9.0,mid_11_20,0
4,1971,1,632,1971-03-06,23,101,12.6,12.5,back_21+,0


### 🚫 Behandlung fehlender Werte


In [142]:
feature_df["driver_recent_form"] = feature_df["driver_recent_form"].fillna(feature_df["driver_recent_form"].mean())
feature_df["constructor_strength"] = feature_df["constructor_strength"].fillna(feature_df["constructor_strength"].mean())

feature_df["driver_experience"] = feature_df["driver_experience"].fillna(0)


feature_df["grid_bucket"] = feature_df["grid_bucket"].astype("object").fillna("unknown")

print(feature_df.isna().mean().sort_values(ascending=False).head(10))

season                  0.0
circuit_name            0.0
grid_bucket             0.0
era                     0.0
constructor_strength    0.0
driver_recent_form      0.0
driver_experience       0.0
grid                    0.0
circuitId               0.0
round                   0.0
dtype: float64


In [143]:
feature_df.to_csv("../data/processed/driver_race_features.csv", index=False)
print("Saved: ../../data/processed/driver_race_features.csv", feature_df.shape)

Saved: ../../data/processed/driver_race_features.csv (25121, 17)
